In [1]:
import math 
from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

In [2]:
import pandas as pd
df = pd.read_csv("../data/points.csv")

nodes = {
   row.node: (row.x, row.y) 
    for row in df.itertuples()
}

In [3]:
# nodes = {
#     0: (0, 0),
#     1: (2, 5),
#     2: (8, 3),
#     3: (5, 9),
#     4: (1, 7),
# }

N = len(nodes)

def distance(i, j):
    x1, y1 = nodes[i]
    x2, y2 = nodes[j]

    return int(math.hypot(x1 - x2, y1 - y2))

In [4]:
manager = pywrapcp.RoutingIndexManager(N,1,0)
routing = pywrapcp.RoutingModel(manager)

In [5]:
def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distance(from_node, to_node)

    
transit_callback_index = routing.RegisterTransitCallback(distance_callback)

In [6]:
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

In [7]:
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
# search_parameters.first_solution_strategy = (
#     routing_enums_pb2.FirstSolutionStrategy.CHRISTOFIDES
# )

In [8]:
solution = routing.SolveWithParameters(search_parameters)
index = routing.Start(0)
route = []

while not routing.IsEnd(index):
    node = manager.IndexToNode(index)
    route.append(node)

    index = solution.Value(routing.NextVar(index))

route.append(
    manager.IndexToNode(index)
)

print(routing.solver().WallTime(), "ms")
print(solution.ObjectiveValue())
print(route)

661 ms
852
[0, 14, 41, 85, 55, 71, 1, 98, 27, 42, 56, 13, 69, 29, 86, 44, 17, 47, 5, 32, 89, 19, 61, 91, 63, 22, 78, 35, 76, 59, 3, 72, 15, 87, 31, 46, 74, 16, 57, 30, 88, 60, 45, 73, 4, 33, 90, 20, 77, 48, 6, 62, 21, 92, 37, 80, 50, 8, 94, 65, 10, 51, 38, 81, 96, 54, 40, 25, 83, 12, 67, 53, 11, 95, 23, 36, 93, 9, 52, 66, 82, 26, 68, 97, 39, 24, 79, 64, 49, 7, 99, 34, 75, 18, 58, 2, 43, 28, 84, 70, 0]


In [9]:
# exact solve results:
#Obj=837.0, walltime: 11.7918 s